> English companion notebook. Code, image paths, and figure references are kept aligned with the Chinese version; the explanation text is rewritten for English readers.

# C61.pdf Technical Review from an Author Perspective: mmLock

Paper: **mmLock: User Leaving Detection Against Data Theft via High-Quality mmWave Radar Imaging**

Authors: Jiawei Xu, Ziqian Bi, Amit Singha, Tao Li, Yimin Chen, Yanchao Zhang  
Venue: 2023 32nd International Conference on Computer Communications and Networks (ICCCN)  
DOI: 10.1109/ICCCN58024.2023.10230151

This notebook reads the paper as an engineering project rather than as a line-by-line translation. The goal is to explain what problem mmLock solves, why FMCW mmWave radar is used, how raw radar samples become point clouds, why three FFT stages are needed, and how PointNet plus temporal modeling turns those point clouds into a user-leaving detector.

Read this notebook together with:

- `radar_fft_cube_progress_en.ipynb`: the signal-processing path from ADC samples to range-Doppler-angle data.
- `cnn_blstm_pointcloud_training_en.ipynb`: the model-training path from point-cloud sequences to classification.
- `C61.pdf`: the original paper.


## Figure Extraction Notes

The PDF figures were extracted into the `img/` directory so the notebook can point to stable local assets:

- `img/embedded/`: raster images directly extracted from the PDF.
- `img/pages/`: full-page PNG renderings of the ten PDF pages, useful when a figure is vector-based or when the extracted image is incomplete.
- Some extracted images are duplicates because PDF extraction can capture the same visual object more than once. When two images are visually identical, the explanation should treat them as one figure instead of repeating the same point.

A practical reading rule: use extracted embedded images when they are clear; use full-page screenshots when a figure is a vector diagram or depends on surrounding captions.


## 1. One-Sentence Summary

mmLock detects whether the legitimate computer user has left the workstation by observing body motion with FMCW mmWave radar. It does not need a camera, a wearable device, or active user interaction. The radar repeatedly measures reflections from the user, turns those reflections into a 3D point cloud, and uses a neural model to decide whether the authorized user is still present or has moved away.

The security scenario is simple: if a user leaves a logged-in computer, a nearby person may try to access private data before the session locks. mmLock tries to shorten that unsafe window by detecting the physical leaving action itself.


## 2. Paper Structure Overview

The paper can be read as four connected layers:

1. **Security motivation**: why screen-lock timeouts, cameras, and wearable authentication are not enough for this situation.
2. **Radar sensing**: why mmWave FMCW radar can observe a person without recording visual identity.
3. **Signal processing**: how raw ADC samples are converted into a range-Doppler-angle representation and then into a point cloud.
4. **Learning system**: how point-cloud sequences are fed into PointNet and LSTM-style temporal modeling to identify the leaving behavior.

For engineering work, the most important bridge is between layer 3 and layer 4. The neural network does not consume raw radio waves directly. It consumes structured points that already encode estimated distance, velocity, angle, and reflection strength.


## 3. System Figure: Main mmLock Pipeline

The system pipeline is: radar capture -> signal processing -> point-cloud generation -> preprocessing -> sequence model -> leave/stay decision.

The important idea is that mmLock is not a single black-box neural network. A large part of the system is deterministic radar processing. The model only becomes meaningful after the radar cube has been transformed into a stable point-cloud sequence.

When reading the figure, track what changes at each stage:

- The radar first records complex ADC samples from multiple RX antennas.
- FFT processing separates reflections by distance, velocity, and direction.
- Detection and clustering keep the points that likely belong to the human body.
- The model reads several consecutive frames so it can recognize motion, not just a static pose.


## 4. Abstract: Core Problem to Explain

The abstract compresses the paper into one claim: high-quality mmWave imaging can detect user leaving in a privacy-preserving and contactless way.

The phrase **high-quality imaging** matters. mmWave radar does not produce an RGB photo, but it can produce a spatial representation of reflecting points. For user-leaving detection, the model does not need to know the face or clothing. It needs enough geometry and motion to distinguish a real leaving trajectory from sitting, turning, or a nearby attacker.

The abstract also signals two evaluation questions:

- Can the system detect leaving reliably under different positions, speeds, and environments?
- Can it still identify the legitimate user's leaving action when another person is nearby?


## 5. Introduction: Why the Problem Matters

The introduction starts from a common security gap: authenticated sessions often remain open after the user physically walks away. Traditional locks rely on a timeout, a manual shortcut, or a proximity device. All of these can fail in ordinary use.

From a sensing perspective, the key observation is that leaving is a physical motion pattern. A user changes distance, angle, and body pose over time. A radar can measure those changes continuously without asking the user to do anything.

This turns access control into a real-time sensing problem: instead of asking whether the password is correct, the system asks whether the authenticated body is still at the workstation.


## 6. Introduction: Why Existing Methods Are Not Enough

The paper contrasts mmLock with several common directions:

- **Timeout-based locking** is simple but slow. A long timeout leaves a security window; a short timeout interrupts normal work.
- **Camera-based monitoring** can capture rich visual information, but it raises privacy concerns and depends on lighting and line of sight.
- **Wearable or phone-based proximity** requires the user to carry a device and can fail if the device is left on the desk.
- **Keyboard and mouse behavior** only sees interaction with the computer, not the user's physical departure.

The paper's argument is not that radar is perfect. The argument is that radar has a useful tradeoff: it senses motion and geometry while avoiding many problems of cameras and wearables.


## 7. Introduction: Why mmWave Radar Fits This Problem

FMCW mmWave radar is useful here because it can measure several physical quantities at once:

- **Range**: how far a reflector is from the radar.
- **Doppler velocity**: whether the reflector is moving toward or away from the radar.
- **Angle**: the direction of arrival across the antenna array.
- **Reflection intensity**: how strong the returned signal is.

These quantities are exactly what a leaving detector needs. A person leaving a chair creates a changing 3D pattern: the torso and limbs move, the body shifts direction, and the distance profile changes. Radar captures this as a sequence of point clouds rather than as a visual image.

This is also where Wi-Fi sensing is conceptually related. Both Wi-Fi and radar observe how wireless signals change after interacting with the human body. The difference is that FMCW radar is designed to estimate range and velocity directly, while Wi-Fi sensing often relies on channel-state changes from communication signals.


## 8. Threat Model

The threat model focuses on data theft after the legitimate user leaves. The attacker is physically near the workstation and tries to use the active session. mmLock attempts to detect the leaving event quickly enough to trigger protection.

A useful way to read the threat model is to separate identities and bodies:

- The **legitimate user** is the person who should be tracked after login.
- The **attacker** may be present nearby and may move differently.
- The system must avoid confusing a nearby attacker with the legitimate user's continued presence.

This is why the paper later discusses nearby-attacker experiments. A leave detector is not enough if it simply sees that some human body remains near the desk.


## 9. Point Cloud Generation: From Radar Signal to Points

The radar does not directly output a point cloud. It outputs sampled beat signals from receiver antennas. Point-cloud generation is the process of converting those samples into estimated object points.

At a high level:

1. The radar transmits FMCW chirps.
2. Reflected chirps return from the body and surrounding objects.
3. Mixing transmitted and received signals produces beat frequencies.
4. FFT processing maps those frequencies into range, velocity, and angle bins.
5. Peak detection selects bins likely to correspond to real reflectors.
6. Each selected bin becomes a point with coordinates and attributes.

The point cloud is therefore already a compressed interpretation of the raw signal. It keeps the parts that look like meaningful physical reflections and drops most of the raw waveform detail.


## 10. Why Three FFT Stages Are Needed

The three FFT stages answer three different questions about the same reflected energy:

- **Range FFT**: at what distance did the reflection occur?
- **Doppler FFT**: how fast is that reflection moving across chirps?
- **Angle FFT**: from which direction did the reflection arrive across antennas?

One FFT is not enough because each dimension is encoded along a different axis of the sampled radar data. Fast-time samples within one chirp encode range. Repeated chirps over time encode Doppler. Multiple antennas encode spatial phase differences for angle.

After these FFTs, a reflection is no longer just a waveform sample. It becomes a candidate physical point in a 3D sensing space.


## 11. First FFT: Why Range FFT Gives Distance

In FMCW radar, the transmitted chirp sweeps frequency over time. A farther object returns a more delayed echo. After mixing the received echo with the current transmitted signal, that time delay becomes a beat frequency.

Range FFT separates these beat frequencies. A peak at a certain frequency bin corresponds to a certain time delay, and the time delay corresponds to distance.

This is the first major simplification: many raw ADC samples become a spectrum where strong peaks indicate likely distances. In code, this is usually the FFT along the ADC-sample axis.


## 12. Second FFT: Why Doppler FFT Gives Velocity

Range FFT tells us where energy is in distance, but leaving is a motion problem. To estimate velocity, the radar sends many chirps and looks at how the phase changes from chirp to chirp.

If a body part is moving, the reflected signal phase evolves across repeated chirps. Doppler FFT separates these phase-change rates and maps them to velocity bins.

This matters for mmLock because a standing-up or walking-away motion is not just a new position. It has temporal dynamics. Doppler features help distinguish leaving from small seated movements.


## 13. Third FFT: Why Angle FFT Gives Direction

Multiple RX antennas receive the same reflection with slightly different phases. The phase difference depends on the direction from which the reflection arrives.

Angle FFT uses the antenna dimension to estimate direction of arrival. With MIMO, multiple TX and RX combinations create a larger virtual antenna array, improving angular resolution.

The result is that a reflection can be localized not only by distance and velocity, but also by direction. This is what allows the system to build a spatial point cloud around the workstation.


## 14. Data Shape After Three FFT Stages

After the FFT stages, the radar data can be viewed as a multidimensional cube. Typical axes include range bins, Doppler bins, and angle bins, with complex values or magnitudes indicating reflection strength.

A point-cloud detector then searches this cube for strong, meaningful peaks. Each selected peak can be converted into a point such as:

- x/y/z coordinates or range/angle representation.
- Doppler velocity.
- signal intensity.
- optional frame index or timestamp.

This conversion is the bridge between signal processing and machine learning.


## 15. MIMO: Why a Virtual Antenna Array Is Needed

TX means **transmitter**: the antenna element that sends the radar chirp. RX means **receiver**: the antenna element that listens for reflections.

In a MIMO radar, different TX antennas transmit in a controlled sequence, and multiple RX antennas record the echoes. Each TX-RX pair behaves like a measurement from a different spatial location. Combining these pairs forms a **virtual antenna array**.

The virtual array is important because angle estimation depends on spatial aperture. More effective antenna positions usually mean better ability to distinguish reflections from different directions. In mmLock, this helps separate the user's body from nearby objects and from another person.


## 16. Raw Point Clouds and Denoising
Raw point clouds include useful body reflections, but also clutter from furniture, walls, static objects, and noise. The paper therefore needs preprocessing before model input.
Denoising usually does three things:
- Remove points that are too weak or outside the region of interest.
- Suppress static background that does not belong to the moving user.
- Keep a stable number of points per frame so the neural model receives a consistent tensor shape.
This step is easy to underestimate. If preprocessing is unstable, the model learns artifacts of the sensing pipeline rather than the leaving behavior.
![Raw radar point cloud](img/embedded/pdf_image-002-003.png)
This figure should be read as the bridge between signal processing and learning. Each visible point is not a pixel from a camera; it is a radar reflection selected after FFT processing and detection. The scattered shape is exactly why preprocessing matters: the model needs the human-motion structure, while clutter and isolated points should be suppressed.


## 17. Range Heatmap Preprocessing
A range heatmap summarizes reflection strength across distance and time or across distance and angle. It is useful because it shows whether the radar sees a structured body motion rather than random noise.
For user-leaving detection, the heatmap can reveal the motion trace: a person moving away changes the dominant range region over consecutive frames. The point cloud gives sparse geometry; the heatmap gives a denser view of signal energy.
In a project implementation, heatmaps are also useful for debugging. If the heatmap does not show clear human motion, the model is unlikely to work well regardless of network architecture.
![Range heatmap example](img/embedded/pdf_image-005-006.png)
A range heatmap gives a denser view of reflected energy than a sparse point cloud. For leaving detection, it helps verify whether the body produces a coherent distance-changing trace over time. If this plot is noisy or flat, the issue is likely in capture setup, calibration, or FFT processing rather than in the classifier.


## 18. Point-Cloud Preprocessing: DBSCAN and Point Count Normalization
Point-cloud frames often contain a variable number of detected points. Neural networks usually prefer fixed-size tensors. The preprocessing stage therefore clusters, filters, and pads or samples points.
DBSCAN is useful because it groups nearby points without requiring a fixed number of clusters. In this context, it can help keep the cluster corresponding to the user and remove isolated noise points.
After clustering, the system normalizes the number of points. If a frame has too many points, it samples or selects a subset. If it has too few, it pads with zeros or repeated placeholders. This makes the input shape stable while preserving the main motion structure.
![Human-shaped radar point cloud](img/embedded/pdf_image-002-004.png)
This kind of point-cloud view is useful because it shows what the model actually receives after radar processing. The point locations roughly describe body geometry and pose, but the frame still needs clustering, filtering, and fixed-size normalization before it can become a stable tensor.


## 19. Why PointNet + LSTM Is Used

PointNet is designed for unordered point sets. A point cloud frame does not have a natural pixel grid like an image, so PointNet-style processing is a reasonable fit for extracting per-frame spatial features.

However, leaving is not a single-frame event. A person may sit, stand, rotate, step away, or pass through intermediate states. LSTM-style temporal modeling reads a sequence of frame features and learns how the pattern evolves over time.

The combination is therefore natural:

- PointNet handles the geometry of each frame.
- LSTM or BiLSTM handles the temporal motion across frames.
- The classifier predicts whether the sequence indicates user leaving.


## 20. Tracking the Target User When an Attacker Is Present
A nearby attacker complicates the problem because the radar may see more than one human body. The system must avoid treating any remaining person as the legitimate user.
The paper's design relies on motion history and spatial continuity. The legitimate user has a tracked point-cloud sequence before and during leaving. An attacker who enters or remains nearby should produce a different trajectory or cluster.
This is why single-frame classification is risky. The model and preprocessing need temporal context so they can identify which body is moving away from the workstation.
![Nearby attacker point-cloud example](img/embedded/pdf_image-006-008.png)
This figure belongs to the security-specific difficulty of mmLock. The radar may observe more than one body, so the task is not simply human-presence detection. The system must preserve the target user's motion history and avoid mistaking another nearby person for the legitimate user.


## 21. System Implementation: Hardware and Capture Parameters
The implementation section connects the method to actual radar hardware. The radar board determines important limits: chirp bandwidth affects range resolution, the number of chirps affects Doppler estimation, and antenna layout affects angle estimation.
When reproducing the work, capture parameters should be documented carefully:
- ADC sample count and sampling rate.
- chirps per frame and frame rate.
- TX/RX antenna count and TDM-MIMO pattern.
- range limits and region of interest.
- dataset labeling rules.
Small parameter changes can alter the radar cube shape and therefore the downstream point-cloud format.
![Experiment setup](img/embedded/pdf_image-006-009.png)
The setup image matters because radar performance depends on physical placement. Radar height, pointing direction, desk layout, and user position all affect range, angle, multipath, and clutter. These are not cosmetic details; they are part of the sensing pipeline.


## 22. Model Training: PointNet Pretraining

The paper separates feature learning from temporal recognition. PointNet can be trained to extract stable spatial features from point-cloud frames, and the temporal model then uses these features over a sequence.

This staged view is helpful for debugging. If PointNet cannot learn meaningful frame-level representations, the sequence model will not fix the problem. Conversely, if frame features are reasonable but predictions are unstable, the issue may be sequence length, labeling, or temporal modeling.

The companion training notebook follows this idea by treating each action sample as consecutive point-cloud frames.


## 23. Training Curves: PointNet and LSTM

Training curves should be read as diagnostics, not just as final scores. A useful curve answers several questions:

- Does training loss decrease smoothly?
- Does validation accuracy improve, or does the model only memorize training data?
- Does the temporal model improve over frame-only features?
- Are errors concentrated in similar actions, positions, or environments?

For mmLock, validation under different leave angles, locations, speeds, and nearby-attacker settings is more important than a single headline accuracy number.


## 24. Single-User Experiment Results

The single-user experiments test the base case: one legitimate user, no nearby attacker. This establishes whether the radar and model can detect leaving under controlled but realistic conditions.

The key interpretation is not just whether the classifier is accurate. It is whether detection happens fast enough to reduce the unsafe access window while avoiding false locks during normal seated behavior.

If this setting fails, the more complex attacker setting cannot be trusted.


## 25. Leaving Angle, Position, Speed, Height, and Environment

A robust leaving detector should not depend on one exact posture or desk setup. The paper therefore varies factors that affect radar reflections:

- leaving direction changes the angle trajectory;
- initial position changes the body location in the radar coordinate system;
- walking speed changes Doppler and sequence duration;
- user height changes the distribution of body points;
- environment changes clutter and multipath.

These experiments test whether the model learns the concept of leaving rather than memorizing one recording condition.


## 26. Nearby-Attacker Experiments

The nearby-attacker setting is the security-specific part of the evaluation. It asks whether mmLock still recognizes the legitimate user's leaving when another person is present.

The hard case is not merely detecting humans. The hard case is maintaining the identity of the target motion sequence. If the legitimate user leaves while an attacker remains close to the computer, a naive occupancy detector might falsely conclude that the user is still present.

This is where point-cloud tracking and temporal modeling become essential.


## 27. Related Work Interpretation

The related-work section places mmLock among continuous authentication, user-presence detection, wireless sensing, and radar-based behavior recognition.

The useful comparison is with Wi-Fi sensing. Wi-Fi CSI can sense human motion by observing channel changes, but the signal is originally designed for communication. FMCW radar is built for sensing and naturally provides range and Doppler structure. This does not make radar universally better, but it explains why radar is a strong fit for this paper's geometry-and-motion task.

The project documentation should make this distinction clear because both methods belong to the broader field of wireless sensing.


## 28. Conclusion Interpretation

The conclusion claims that mmLock can reduce the risk of post-departure data theft by detecting user leaving with mmWave radar. The important engineering takeaway is the full stack: hardware capture, FFT processing, point-cloud generation, preprocessing, temporal modeling, and security evaluation all have to work together.

A weak implementation at any layer can break the system. For example, poor radar calibration gives noisy point clouds; unstable clustering gives inconsistent model input; incomplete evaluation hides attacker cases.

This is why the project should document both the paper idea and the processing code path.


## 29. Mapping to the Engineering Code

Use the companion files as the practical path through the project:

- `radar_fft_cube_progress_en.ipynb`: explains the radar-processing side from raw ADC data to FFT cube and detected points.
- `cnn_blstm_pointcloud_training_en.ipynb`: explains the model-training side from point-cloud sequences to CNN + BiLSTM classification.
- `radar_fft_cube_progress_zh.ipynb` and `cnn_blstm_pointcloud_training_zh.ipynb`: Chinese versions for the same workflows.

External reference implementations are also linked in the README and site:

- Radar processing platform: https://github.com/billzi2016/mmwave-fmcw-cascade-mimo-sensing-platform
- Radar simulator: https://github.com/billzi2016/MIMO-FMCW-Radar-Simulator-Multiprocess

The paper explains why the system should work. The notebooks explain how the data is shaped so the model can actually learn from it.
